In [ ]:
import sys, os, glob, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
if not torch.cuda.is_available(): raise SystemExit("GPU не выделена")
name = torch.cuda.get_device_name(0); print("GPU:", name)
# P100 (sm_60) в нынешней сборке torch на Kaggle не поддерживается: любой прогон падает
# с "no kernel image is available for execution on the device". Падаем сразу, не тратя час.
if "T4" not in name and "L4" not in name and "A100" not in name:
    raise SystemExit(f"нужна T4/L4, выдали {name}")
modules = glob.glob("/kaggle/input/**/train_ce_large.py", recursive=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
for path in glob.glob(os.path.dirname(modules[0]) + "/*.py"):
    shutil.copy(path, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
pack = os.path.dirname(glob.glob("/kaggle/input/**/item_texts.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/pack", exist_ok=True)
for path in glob.glob(pack + "/*"):
    dst = "/kaggle/working/pack/" + os.path.basename(path)
    if not os.path.exists(dst): os.symlink(path, dst)
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
sys.argv = ["train_ce_large", "--prepacked", "/kaggle/working/pack", "--holdout-fold", "0",
            "--base-model", "intfloat/multilingual-e5-large",
            "--epochs", "1", "--batch-size", "64", "--max-length", "256",
            # 3e-5 крупную основу e5 разносит: loss упал до 0.40, а на пике
            # расписания (шаг 1500) подскочил до 0.52 и там остался.
            "--learning-rate", "1e-5",
            "--max-train-pairs", "400000",
            "--output", "/kaggle/working/ce_e5large"]
from src.train_ce_large import main
main()
